# Derive a single or set of best performing model configurations for confirmatory analysis

## Set Up

In [ ]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
import pandas as pd
from src.perm_res import (
    get_avg_r2,
    get_layer1,
    get_layer2,
    get_layer3,
    get_raw_p_df,
    get_collapsed_null,
    get_corr_p_vals,
)

raw_preds_dir = BASE_PATH / "results/perm_res/preds/baseline"
r2_dir = BASE_PATH / "results/comprehensive_r2"
compare_dir = BASE_PATH / "results/compare"

In [ ]:
parent_base_df = pd.read_parquet(raw_preds_dir / "parent/pred_df.parquet")
self_base_df = pd.read_parquet(raw_preds_dir / "self/pred_df.parquet")

## Load preds

In [ ]:
r2_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
r2_df_parent = get_avg_r2(parent_base_df)
r2_1_parent = get_layer1(r2_df_parent)
print("1 layer:")
display(r2_1_parent)
# 2
r2s_2_S_base_parent, r2s_2_Q_base_parent = get_layer2(r2_df_parent, filter_best_Q=True)
print("2 layer S base:")
display(r2s_2_S_base_parent)
print("2 layer Q base:")
display(r2s_2_Q_base_parent)
# 3
r2_3_parent = get_layer3(r2_df_parent, filter_best_Q=True)
print("3 layer:")
display(r2_3_parent)
##Export
r2_df_parent.to_parquet(r2_dir / "parent.parquet")

In [ ]:
r2_df_self = get_avg_r2(self_base_df)
# 1
r2_1_self = get_layer1(r2_df_self)
display(r2_1_self)
# 2
r2s_2_S_base_self, r2s_2_Q_base_self = get_layer2(r2_df_self, filter_best_Q=True)
display(r2s_2_S_base_self)
display(r2s_2_Q_base_self)
# 3
r2_3_self = get_layer3(r2_df_self, filter_best_Q=True)
display(r2_3_self)
##Export
r2_df_self.to_parquet(r2_dir / "self.parquet")

## Compare models

Bayesian comparison

In [ ]:
parent_r2 = pd.read_parquet(r2_dir / "parent.parquet")
self_r2 = pd.read_parquet(r2_dir / "self.parquet")

In [ ]:
params = [
    "num_layers",
    "agg_type",
    "llm_model",
    "desc_seg",
    "alpha",
    "two_layer_base",
    "temp",
]
from sklearn.metrics import mean_absolute_error, r2_score
from baycomp import two_on_single

SCORE_TYPE = "r2"
compare_dir.mkdir(exist_ok=True, parents=True)
exp_path_parent = compare_dir / f"parent_{SCORE_TYPE}.parquet"
exp_path_self = compare_dir / f"self_{SCORE_TYPE}.parquet"

### Calculate Bayesian Compare Vals

In [ ]:
def get_top_models(r2_df, params, top_n=50):
    r2_3_layer = get_layer3(r2_df, filter_best_Q=True)
    r2_3_layer = r2_3_layer.sort_values(by="diff", ascending=False)
    # Take top 50 model configs
    r2_3_top = r2_3_layer.iloc[:top_n]
    top_model_params = r2_3_top.iloc[0][params]
    return r2_3_top, top_model_params


def get_score(pred_df, param_vals, score_type):
    """
    df: pandas DF
        Contains predictions for all models
    param_vals: pd.Series
        Mapping params to specific values for filtering
    type: str
        One of ['mae', 'r2']
    """
    df_sub = pred_df[(pred_df[param_vals.keys().to_list()] == param_vals).all(axis=1)]
    if score_type == "mae":
        score_df_fold_repeats = (
            df_sub.groupby(["repeat", "split_id"])
            .apply(lambda g: mean_absolute_error(g["pred_stage3_total"], g["true"]))
            .rename("mae")
            .reset_index()
        )
        score_df_repeats = (
            df_sub.groupby(["repeat"])
            .apply(lambda g: mean_absolute_error(g["pred_stage3_total"], g["true"]))
            .rename("mae")
            .reset_index()
        )
        score_df_fold_repeats_q = (
            df_sub.groupby(["repeat", "split_id"])
            .apply(lambda g: mean_absolute_error(g["pred_stage2_total"], g["true"]))
            .rename("mae")
            .reset_index()
        )
        score_df_repeats_q = (
            df_sub.groupby(["repeat"])
            .apply(lambda g: mean_absolute_error(g["pred_stage2_total"], g["true"]))
            .rename("mae")
            .reset_index()
        )

    elif score_type == "r2":
        # display(df_sub)
        score_df_fold_repeats = (
            df_sub.groupby(["repeat", "split_id"])
            .apply(lambda g: r2_score(y_pred=g["pred_stage3_total"], y_true=g["true"]))
            .rename("r2")
            .reset_index()
        )
        score_df_repeats = (
            df_sub.groupby(["repeat"])
            .apply(lambda g: r2_score(y_pred=g["pred_stage3_total"], y_true=g["true"]))
            .rename("r2")
            .reset_index()
        )
        score_df_fold_repeats_q = (
            df_sub.groupby(["repeat", "split_id"])
            .apply(lambda g: r2_score(y_pred=g["pred_stage2_total"], y_true=g["true"]))
            .rename("r2")
            .reset_index()
        )
        score_df_repeats_q = (
            df_sub.groupby(["repeat"])
            .apply(lambda g: r2_score(y_pred=g["pred_stage2_total"], y_true=g["true"]))
            .rename("r2")
            .reset_index()
        )

    return (
        score_df_fold_repeats,
        score_df_repeats,
        score_df_fold_repeats_q,
        score_df_repeats_q,
    )


def compare_model_configs(base_pred_df, r2_df, param_list, top_n_models, score_type):
    r2_3_top, top_model_params = get_top_models(r2_df, param_list, top_n=top_n_models)
    best_score_df_rf, _, _, _ = get_score(
        base_pred_df, top_model_params, score_type=score_type
    )

    new_rows = []
    for i, row in r2_3_top.reset_index(drop=True).iterrows():
        new_row = row[params].to_dict()
        cur_score_df_rf, cur_score_df_r, cur_score_df_rf_q, cur_score_df_r_q = (
            get_score(base_pred_df, row[param_list], score_type=score_type)
        )
        if i == 0:
            new_row.update(
                {
                    "P(best better)": 0.0,
                    "P(equal)": 1.0,
                    "P(this better)": 0.0,
                }
            )
        else:
            combined_df = pd.merge(
                best_score_df_rf,
                cur_score_df_rf,
                on=["repeat", "split_id"],
                suffixes=("_best", "_cur"),
            )
            # (p_left, p_rope, p_right), fig = two_on_single(
            p_left, p_rope, p_right = two_on_single(
                x=combined_df[f"{score_type}_best"],
                y=combined_df[f"{score_type}_cur"],
                rope=0.01,
                runs=10,
                plot=False,
            )
            new_row.update(
                {
                    "P(best better)": round(p_left, 2),
                    "P(equal)": round(p_rope, 2),
                    "P(this better)": round(p_right, 2),
                }
            )
        # average across n_repeats * n_folds values
        avg_r2_rf = float(cur_score_df_rf[score_type].mean())
        std_r2_rf = float(cur_score_df_rf[score_type].std())
        avg_r2_rf_q = float(cur_score_df_rf_q[score_type].mean())
        std_r2_rf_q = float(cur_score_df_rf_q[score_type].std())
        # average across n_repeats values (each val calculated within a repeat across folds)
        avg_r2_r = float(cur_score_df_r[score_type].mean())
        std_r2_r = float(cur_score_df_r[score_type].std())
        avg_r2_r_q = float(cur_score_df_r_q[score_type].mean())
        std_r2_r_q = float(cur_score_df_r_q[score_type].std())
        new_row.update(
            {
                f"{score_type} repeats-folds": f"{avg_r2_rf:.3f} ({std_r2_rf:.3f})",
                f"{score_type} repeats": f"{avg_r2_r:.3f} ({std_r2_r:.3f})",
                f"{score_type} repeats-folds- Q": f"{avg_r2_rf_q:.3f} ({std_r2_rf_q:.3f})",
                f"{score_type} repeats- Q": f"{avg_r2_r_q:.3f} ({std_r2_r_q:.3f})",
            }
        )
        new_rows.append(new_row)

    compare_df = pd.DataFrame(new_rows)
    return compare_df

In [ ]:
compare_df_parent = compare_model_configs(
    base_pred_df=parent_base_df,
    r2_df=parent_r2,
    param_list=params,
    top_n_models=50,
    score_type=SCORE_TYPE,
)
if exp_path_parent.exists():
    exp_path_parent.unlink()
compare_df_parent.to_parquet(exp_path_parent)


compare_df_self = compare_model_configs(
    base_pred_df=self_base_df,
    r2_df=self_r2,
    param_list=params,
    top_n_models=50,
    score_type=SCORE_TYPE,
)
if exp_path_self.exists():
    exp_path_self.unlink()
compare_df_self.to_parquet(exp_path_self)

In [ ]:
display(
    compare_df_parent.sort_values("P(equal)", ascending=False)
)  # 12 that are ~equal--> scale/llama/few_shot/15_(15-25)/temp=(0-1)
display(
    compare_df_self.sort_values("P(equal)", ascending=False)
)  # only 1 equal -->scale/gemini/few_shot/50_15/temp=0.4

### Utilize vals for compare/selection

- Via correlation cluster mapping

In [ ]:
compare_df_parent = pd.read_parquet(exp_path_parent)
compare_df_self = pd.read_parquet(exp_path_self)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


def plot_scatter(error_best, error_cur, i):
    fig, ax = plt.subplots()
    ax.scatter(
        error_best,
        error_cur,
        label="prediction error per participant in each fold/repeat",
        s=20,
        alpha=0.7,
    )
    ax.set_xlabel("error_model_0")
    ax.set_ylabel(f"error_model_{i}")
    plt.show()


def get_cur_pred_error(pred_df, i, row):
    mask = (pred_df[params] == row[params]).all(axis=1)
    cur_preds = pred_df[mask].reset_index(drop=True)

    label = f"error_model_{i}"
    cur_preds[label] = cur_preds["true"] - cur_preds["pred_stage3_total"]

    cur_preds_sub = cur_preds[["DAWBA_ID", "repeat", "split_id", label]].set_index(
        ["DAWBA_ID", "repeat", "split_id"]
    )
    return cur_preds_sub


def plot_cluster(compare_df, pred_df, prob_threshold):
    inconclusive_mask = (
        compare_df[["P(best better)", "P(equal)", "P(this better)"]] < prob_threshold
    ).all(axis=1)
    conclusive_mask = compare_df["P(equal)"] > prob_threshold
    eq_param_df = compare_df[inconclusive_mask | conclusive_mask]

    display(eq_param_df)
    model_frames = []
    for i, (idx, row) in enumerate(eq_param_df.iterrows()):
        cur_preds_sub = get_cur_pred_error(pred_df, i, row)
        model_frames.append(cur_preds_sub)
        if i != 0:
            plot_scatter(model_frames[0], cur_preds_sub, i)

    residual_wide = pd.concat(model_frames, axis=1, join="outer")  # instances x models
    residual_corr = (
        residual_wide.corr()
    )  # pairwise-complete by default, robust to misaligned folds
    sns.clustermap(
        residual_corr, cmap="viridis", annot=True, fmt=".2f", vmin=0.97, vmax=1
    )
    return residual_corr

In [ ]:
residual_corr_parent = plot_cluster(
    compare_df_parent, parent_base_df, prob_threshold=0.5
)

In [ ]:
residual_corr_self = plot_cluster(compare_df_self, self_base_df, prob_threshold=0.5)